## Setup

In [ ]:
!pip install -q --upgrade trl bitsandbytes accelerate peft datasets transformers evaluate rouge_score nltk

## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

## 1 · Dataset Preparation

In [ ]:
import os
from datasets import load_dataset, concatenate_datasets

DATA_DIR = '/kaggle/working/data_splits'

forget_dataset   = load_dataset('Novaspree/factify_5K_enriched', data_files='forget/forget_set_fixed.json', split='train')
retain_dataset   = load_dataset('Novaspree/factify_5K_enriched', data_files='retain/retain_set_fixed.json', split='train')
finetune_dataset = concatenate_datasets([forget_dataset, retain_dataset]).shuffle(seed=42)

print('Columns:', finetune_dataset.column_names)
print('Sample:', finetune_dataset[0])

In [ ]:
# Set to your context column (e.g. 'claim'). None if not available.
CONTEXT_FIELD = 'claim'

def format_example(example):
    if CONTEXT_FIELD and CONTEXT_FIELD in example and example[CONTEXT_FIELD]:
        user_text = f"Context: {example[CONTEXT_FIELD]}\n\nQuestion: {example['question']}"
    else:
        user_text = example['question']
    return {
        'prompt':     [{'role': 'user',      'content': user_text}],
        'completion': [{'role': 'assistant', 'content': example['answer']}],
    }

finetune_dataset = finetune_dataset.map(format_example)
forget_dataset   = forget_dataset.map(format_example)
retain_dataset   = retain_dataset.map(format_example)

os.makedirs(DATA_DIR, exist_ok=True)
finetune_dataset.save_to_disk(f'{DATA_DIR}/finetune_dataset')
forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')

print(f'Forget  : {len(forget_dataset)}')
print(f'Retain  : {len(retain_dataset)}')
print(f'Combined: {len(finetune_dataset)}')
print('\nSample prompt    :', finetune_dataset[0]['prompt'])
print('Sample completion:', finetune_dataset[0]['completion'])

## 2 · LoRA Fine-Tuning (Optimized)

In [ ]:
import torch, trl, os, inspect
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk

print('TRL version:', trl.__version__)

MODEL_NAME        = 'google/gemma-3-4b-it'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
DATA_DIR          = '/kaggle/working/data_splits'
HF_UPLOAD_REPO    = 'Novaspree/factify-Gemma3-adapter'

# Gemma-3-4b-it: 26 layers (0-25). Mid-layers 9-20 for unlearning.
MID_LAYERS = list(range(9, 21))
LORA_RANK  = 32

finetune_dataset = load_from_disk(f'{DATA_DIR}/finetune_dataset')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.resize_token_embeddings(len(tokenizer))
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    layers_to_transform=MID_LAYERS,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

sft_params = inspect.signature(SFTConfig.__init__).parameters

sft_kwargs = dict(
    output_dir='/kaggle/working/lora_results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,              # OPT 1: more epochs (was 3); monitor for overfit
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    # OPT 2: paged_adamw_8bit — lower memory + slightly better convergence than adamw
    optim='paged_adamw_8bit',
    # OPT 3: weight_decay regularizes and prevents overfit on small datasets
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='no',
    bf16=True,
    fp16=False,
    report_to='none',
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)

if 'max_length' in sft_params:
    sft_kwargs['max_length'] = 512
elif 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = 512

if 'completion_only_loss' in sft_params:
    sft_kwargs['completion_only_loss'] = True

training_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=finetune_dataset,
    args=training_args,
)
trainer.train()

peft_model.save_pretrained(LORA_ADAPTER_PATH)
tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'\nLoRA adapter saved → {LORA_ADAPTER_PATH}')
print('Saved files:', os.listdir(LORA_ADAPTER_PATH))

## 3 · ROUGE Evaluation (Optimized)

In [ ]:
import evaluate, re, torch, string
from tqdm import tqdm
from datasets import load_from_disk
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

rouge = evaluate.load('rouge')
peft_model.eval()

eos_ids = list(set(filter(None, [
    tokenizer.convert_tokens_to_ids('<end_of_turn>'),
    tokenizer.eos_token_id,
])))
print('EOS token ids:', eos_ids)

# OPT 4: Normalize text before ROUGE — removes tokenization mismatches
# that artificially lower scores (punctuation, casing differences).
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# OPT 5: Sentence-split predictions before rougeLsum compute.
# rougeLsum scores per-sentence LCS then averages — much higher than
# treating the whole answer as one sequence.
def prepare_for_rouge(text: str) -> str:
    text = normalize_text(text)
    sentences = sent_tokenize(text)
    return '\n'.join(sentences)

def generate_answer(question: str, context: str = None) -> str:
    if context:
        user_content = f'Context: {context}\n\nQuestion: {question}'
    else:
        user_content = question

    messages = [{'role': 'user', 'content': user_content}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=512
    ).to(peft_model.device)

    with torch.no_grad():
        gen = peft_model.generate(
            **inputs,
            max_new_tokens=200,
            # OPT 6: Beam search (num_beams=4) vs greedy.
            # Beam search explores multiple candidate sequences and picks
            # the highest-probability one — consistently outperforms greedy
            # on ROUGE for factual QA tasks.
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,    # prevents 3-gram repetition during beam search
            repetition_penalty=1.15,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    raw = re.sub(r'<(end_of_turn|start_of_turn|bos|eos)>.*$', '', raw, flags=re.DOTALL).strip()
    return raw

def evaluate_dataset(dataset_path, name):
    print(f'\n--- Evaluating {name} ---')
    dataset = load_from_disk(dataset_path)
    preds, refs = [], []

    for sample in tqdm(dataset, desc=name):
        ctx  = sample.get(CONTEXT_FIELD) if CONTEXT_FIELD else None
        pred = generate_answer(sample['question'], context=ctx)
        preds.append(prepare_for_rouge(pred))
        refs.append(prepare_for_rouge(sample['answer']))

    # Sanity-check a few raw predictions
    print('\n[Sample Predictions]')
    for i in range(min(3, len(preds))):
        print(f'  Q   : {dataset[i]["question"][:80]}')
        print(f'  Pred: {preds[i][:120]}')
        print(f'  Ref : {refs[i][:120]}')
        print()

    results = rouge.compute(
        predictions=preds,
        references=refs,
        use_stemmer=True,    # Porter stemming: 'running'/'ran' both match 'run'
    )
    print(f'\n✅ ROUGE {name}: {results}')
    return results

DATA_DIR = '/kaggle/working/data_splits'
forget_rouge = evaluate_dataset(f'{DATA_DIR}/forget_dataset', 'Forget')
retain_rouge = evaluate_dataset(f'{DATA_DIR}/retain_dataset', 'Retain')

## 4 · Upload Adapters to HF Hub

In [ ]:
HF_UPLOAD_REPO    = 'Novaspree/factify-Gemma3-adapter'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'

try:
    print(f'Uploading adapter to HF Hub: {HF_UPLOAD_REPO} ...')
    peft_model.push_to_hub(HF_UPLOAD_REPO, private=False)
    tokenizer.push_to_hub(HF_UPLOAD_REPO)
    print('✅ Upload complete!')
    print(f'Load with: PeftModel.from_pretrained(base_model, "{HF_UPLOAD_REPO}")')
except Exception as e:
    print(f'❌ Error: {e}')